# Impulse and Torque MLP (vertex-transform, no GNN)

This notebook trains a physics-structured and physics-informed MLP for a
rigid body interacting with a plane.

**Variant:** the GCN / GNN is removed. We still place the body's mesh
vertices in world space (rotation + translation, plus per-vertex velocity
from rigid-body kinematics), but those transformed vertices are flattened
and fed **directly into the MLP** — no graph convolutions, no `edge_index`,
no `torch_geometric`.

As input we get the rotation as a 6-D rotation matrix (with sin and cos
of roll / pitch / yaw). We parameterise normal force with Hooke, similar
to our simulations. We internally predict the contact normal. We couple
torque to force via cross product: `torque = r_lever x f`. In the loss we
use a Huber loss with a weight for energy conservation.


This notebook trains a physics-structured and physics informed MLP for a rigid body interacting with a plane-
As input we get the rotation as a 6-D rotation Matrix (with cos and sin)
We parameterise normal force with Hooke, similar to our simulations
We internally predict the contact normal
We couple torque to force via cross product: torque = r_{lever} x f
In the loss we use a Hube loss with a weight for energy conservation

---

**Update — closing the gap to the GNN.** The earlier version of this MLP trained much worse than the GCN variant for two reasons, both now fixed in the `VertexMLPWrench` cell:

1. *Vertex features were crippled.* The GCN feeds each vertex 16 geometric channels (local xyz, world xyz, world z, body linear & angular velocity, and the per-vertex world velocity `v_lin + ω×r`). The MLP was keeping only the 3 world coordinates and silently dropping all velocity information. The full 16-D set is now built per vertex, matching the GCN.

2. *Flatten → pool.* The MLP concatenated a per-sample-variable selection of vertices into one flat vector, so a given physical vertex landed in a different slot on every sample — an alignment the network cannot learn. It now applies a shared per-vertex encoder followed by a permutation-invariant max+mean pool, exactly the reduction the GCN uses after message passing. This is the dominant fix.

The mesh is now decimated to `face_count=1000` to match the GCN notebook.

In [ ]:
%pip install trimesh
%pip install fast-simplification

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 77.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import json
import math
from tqdm import tqdm
import trimesh

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print("Using device:", device)

Using device: cuda


## Constants

In [ ]:
TRAIN_TEST_SPLIT = 0.8

## Colab Only - Download data

In [ ]:
def is_colab():
    try:
        import google.colab
        return True
    except Exception as e:
        return False

if is_colab():
    from google.colab import drive
    from tqdm import tqdm
    import os
    import json
    import shutil

    drive.mount('/content/drive', force_remount=True)

    # --- Copy the JSON file ---
    src_path = "/content/drive/MyDrive/final_output_contact_points.json"
    dst_path = "/content/final_output_contact_points.json"
    chunk_size = 1024 * 1024  # 1 MB
    file_size = os.path.getsize(src_path)

    with open(src_path, 'rb') as src, open(dst_path, 'wb') as dst:
        with tqdm(total=file_size, unit='B', unit_scale=True, desc="Copying JSON to /content") as pbar:
            while True:
                chunk = src.read(chunk_size)
                if not chunk:
                    break
                dst.write(chunk)
                pbar.update(len(chunk))
    print("Done! File is now in:", dst_path)

    # --- Copy the blender_models folder ---
    src_folder = "/content/drive/MyDrive/blender_models"
    dst_folder = "/content/blender_models"

    # Gather all files first so we know the total size for the progress bar
    all_files = []
    total_size = 0
    for root, dirs, files in os.walk(src_folder):
        for f in files:
            full = os.path.join(root, f)
            try:
                size = os.path.getsize(full)
            except OSError:
                size = 0
            all_files.append((full, size))
            total_size += size

    print(f"\nFound {len(all_files)} files in blender_models ({total_size / (1024**2):.1f} MB total)")

    os.makedirs(dst_folder, exist_ok=True)

    with tqdm(total=total_size, unit='B', unit_scale=True, desc="Copying blender_models") as pbar:
        for src_file, size in all_files:
            rel = os.path.relpath(src_file, src_folder)
            dst_file = os.path.join(dst_folder, rel)
            os.makedirs(os.path.dirname(dst_file), exist_ok=True)

            with open(src_file, 'rb') as src, open(dst_file, 'wb') as dst:
                while True:
                    chunk = src.read(chunk_size)
                    if not chunk:
                        break
                    dst.write(chunk)
                    pbar.update(len(chunk))

    print("Done! Folder is now in:", dst_folder)

    # --- Print the first entry of the JSON ---
    with open(dst_path, 'r') as f:
        data = json.load(f)
    first = data[0] if isinstance(data, list) else next(iter(data.values()))
    print("\nFirst entry:")
    print(json.dumps(first, indent=2))

Mounted at /content/drive


Copying JSON to /content: 100%|██████████| 3.06G/3.06G [01:12<00:00, 42.3MB/s]


Done! File is now in: /content/final_output_contact_points.json

Found 10 files in blender_models (23.7 MB total)


Copying blender_models: 100%|██████████| 24.9M/24.9M [00:10<00:00, 2.30MB/s]


Done! Folder is now in: /content/blender_models

First entry:
{
  "run_id": "45607-10962",
  "frame": 1100,
  "self_position": {
    "x": -2.1698737809626805e-10,
    "y": 3.4294756871541397e-10,
    "z": 0.4251065856335491
  },
  "linear_velocity": {
    "x": -1.6333582256566178e-11,
    "y": 1.5419311483248526e-10,
    "z": 0.9608798742678641
  },
  "angular_velocity": {
    "x": 1.845335601592928,
    "y": -0.30634940345629424,
    "z": -0.07498392804647439
  },
  "self_rotation": {
    "qx": 0.07934607860633995,
    "qy": 0.06944786314683987,
    "qz": 0.9942559357458716,
    "qw": 0.01833925702518294,
    "roll": 0.14322769502831242,
    "pitch": -0.15586368752750532,
    "yaw": 3.093502728655825
  },
  "collider_position": {
    "x": 0.0,
    "y": 0.0,
    "z": 0.0
  },
  "collider_rotation": {
    "qx": 0.0,
    "qy": 0.0,
    "qz": 0.0,
    "qw": 1.0,
    "roll": 0.0,
    "pitch": -0.0,
    "yaw": 0.0
  },
  "relative_position_to_collider": {
    "x": -2.1698737809626805e-10,
 

## Dataset

Here we get the contat points with individual forces. From these forces, we calculate the torque and sum up the forces and torques to one force and one torque

In [ ]:
class ContactDataset(Dataset):
    """World-frame wrench dataset for a rigid body on a plane.

    Features (13-D):
        [v_x, v_y, v_z,                           # linear velocity (world frame)
         w_x, w_y, w_z,                           # angular velocity (world frame)
         rel_pos_z,                               # height above plane
         sin(roll), cos(roll),
         sin(pitch), cos(pitch),
         sin(yaw), cos(yaw)]

    Per-sample extras (used by the GCN to place vertices in world space):
        self_position: (3,) world-frame position of the body origin (= COM
                       used in the torque computation: lever = r_world - cube_pos).

    Targets (world frame, PHYSICAL UNITS — not normalised):
        force:  (3,)   sum of per-contact forces
        torque: (3,)   sum of (r_world - com_world) x f_world
    """

    def __init__(self, data_list):
        features, forces, torques, collisions = [], [], [], []
        lin_vels, ang_vels, self_positions = [], [], []

        for contact in data_list:
            rel_pos = contact["relative_position_to_collider"]
            rel_rot = contact["relative_rotation_to_collider"]
            lin_vel = contact["linear_velocity"]
            ang_vel = contact["angular_velocity"]
            cube_pos = contact["self_position"]

            roll, pitch, yaw = rel_rot["roll"], rel_rot["pitch"], rel_rot["yaw"]
            feats = np.array([
                lin_vel["x"], lin_vel["y"], lin_vel["z"],
                ang_vel["x"], ang_vel["y"], ang_vel["z"],
                rel_pos["z"],
                np.sin(roll), np.cos(roll),
                np.sin(pitch), np.cos(pitch),
                np.sin(yaw), np.cos(yaw),
            ], dtype=np.float32)

            cube_pos_numpy = np.array([cube_pos["x"], cube_pos["y"], cube_pos["z"]],
                                      dtype=np.float32)
            force_numpy = np.zeros(3, dtype=np.float32)
            torque_numpy = np.zeros(3, dtype=np.float32)

            for p in contact.get("points", []):
                p_force = p["force"]
                lever_pos = p["contact_position_world"]
                point_force_numpy = np.array(
                    [p_force["x"], p_force["y"], p_force["z"]], dtype=np.float32)
                lever_rel_pos = np.array(
                    [lever_pos["x"], lever_pos["y"], lever_pos["z"]],
                    dtype=np.float32) - cube_pos_numpy

                force_numpy += point_force_numpy
                torque_numpy += np.cross(lever_rel_pos, point_force_numpy)

            features.append(feats)
            forces.append(force_numpy)
            torques.append(torque_numpy)
            collisions.append([min(len(contact.get("points", [])), 1)])
            lin_vels.append([lin_vel["x"], lin_vel["y"], lin_vel["z"]])
            ang_vels.append([ang_vel["x"], ang_vel["y"], ang_vel["z"]])
            self_positions.append(cube_pos_numpy)

        self.features       = torch.FloatTensor(np.asarray(features))
        self.forces         = torch.FloatTensor(np.asarray(forces))
        self.torques        = torch.FloatTensor(np.asarray(torques))
        self.collisions     = torch.FloatTensor(np.asarray(collisions))
        self.lin_vels       = torch.FloatTensor(np.asarray(lin_vels, dtype=np.float32))
        self.ang_vels       = torch.FloatTensor(np.asarray(ang_vels, dtype=np.float32))
        self.self_positions = torch.FloatTensor(np.asarray(self_positions, dtype=np.float32))

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return (
            self.features[idx],
            {
                "force":         self.forces[idx],
                "torque":        self.torques[idx],
                "is_collision":  self.collisions[idx],
                "lin_vel":       self.lin_vels[idx],
                "ang_vel":       self.ang_vels[idx],
                "self_position": self.self_positions[idx],
            },
        )

### Show dataset

show the first entry of the dataset

In [ ]:
json_file = 'final_output_contact_points.json'
with open(json_file, 'r') as f:
    data = json.load(f)
if isinstance(data, dict):
    data = [data]

full_dataset = ContactDataset(data)


Print the first entry of the dataset

In [ ]:
print(full_dataset[0])

(tensor([-1.6334e-11,  1.5419e-10,  9.6088e-01,  1.8453e+00, -3.0635e-01,
        -7.4984e-02,  4.2511e-01,  1.4274e-01,  9.8976e-01, -1.5523e-01,
         9.8788e-01,  4.8071e-02, -9.9884e-01]), {'force': tensor([-1.2913e-12,  1.5546e-12,  4.5457e+00]), 'torque': tensor([-2.8540e+00, -8.0085e-01, -5.3687e-13]), 'is_collision': tensor([1.]), 'lin_vel': tensor([-1.6334e-11,  1.5419e-10,  9.6088e-01]), 'ang_vel': tensor([ 1.8453, -0.3063, -0.0750]), 'self_position': tensor([-2.1699e-10,  3.4295e-10,  4.2511e-01])})


Showing some info of the dataset

In [ ]:
print("collisions:", int(full_dataset.collisions.sum().item()),
      "/", len(full_dataset))
mask = full_dataset.collisions.squeeze(-1).bool()
if mask.any():
    f_rms = full_dataset.forces[mask].pow(2).mean().sqrt().item()
    t_rms = full_dataset.torques[mask].pow(2).mean().sqrt().item()
    print(f"\nContact-only RMS force  = {f_rms:.4f}")
    print(f"Contact-only RMS torque = {t_rms:.4f}")
    print(f"Suggested w_torque / w_force ratio ≈ {f_rms / max(t_rms, 1e-8):.3f}")


collisions: 1118967 / 2148306

Contact-only RMS force  = 329.9146
Contact-only RMS torque = 69.7544
Suggested w_torque / w_force ratio ≈ 4.730


Split the dataset into train and test dataset and create data loaders

In [ ]:
train_size = int(TRAIN_TEST_SPLIT * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator = torch.Generator())

use_gpu = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True,
    num_workers=4 if use_gpu else 0,
    pin_memory=False,
    persistent_workers=False,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=4 if use_gpu else 0,
    pin_memory=False,
    persistent_workers=False,
)

Validation checks on the dataset:

In [ ]:
input_dim = full_dataset.features.shape[1]
print(f"input_dim={input_dim}  (expect 10)")
print(f"force target shape = {tuple(full_dataset.forces.shape)}  (expect (N, 3))")
print(f"torque target shape = {tuple(full_dataset.torques.shape)}  (expect (N, 3))")
assert max(full_dataset.collisions) == 1


input_dim=13  (expect 10)
force target shape = (2148306, 3)  (expect (N, 3))
torque target shape = (2148306, 3)  (expect (N, 3))


## Mesh (local-frame vertices)

We load the mesh and keep only the **vertex positions in the body's local
frame**. There is no graph here — no edges, no neighbours — because the
MLP doesn't operate on a graph. The vertices are a fixed-size, fixed-order
geometric description of the body's shape, and we feed them into the MLP
after rotating them into world space.


In [ ]:
import trimesh
import numpy as np

# We still need the mesh: its vertices (in the local body frame) are the
# geometric input to the MLP after the world-space transform.
mesh = trimesh.load('blender_models/bunny.obj',
                    process=True, force='mesh')
mesh.merge_vertices(merge_tex=True, merge_norm=True)

mesh = mesh.simplify_quadric_decimation(face_count=4000)
mesh.export('blender_models/bunny_simplified.obj')

print(mesh.vertices.shape)

vertice_positions = np.array(mesh.vertices)
num_vertices = len(mesh.vertices)


(2002, 3)


## Model

Physically structured MLP. The vertices are rotated (and translated) into
world space, the per-vertex world-frame velocity is computed, and then
**everything is flattened and fed into the MLP** — no graph convolutions,
no message passing.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

HEAD_OUT_DIM = 11  # 1 (collision) + 1 (depth) + 3 (force_residual) + 3 (normal) + 3 (lever)
VEL_SLICE = slice(0, 3)  # [vx, vy, vz] are the first 3 features


class ResBlock(nn.Module):
    def __init__(self, width, expansion=4):
        super().__init__()
        self.act = nn.ReLU(inplace=True)

        self.block = nn.Sequential(
            nn.LayerNorm(width),
            nn.Linear(width, width * expansion),
            self.act,
            nn.LayerNorm(width * expansion),
            nn.Linear(width * expansion, width),
            self.act,
        )

    def forward(self, x):
        return self.act(x + self.block(x))


class WrenchPredictor(nn.Module):
    """
    Predicts net wrench (force + torque) with a head whose force assembly
    matches the physics sim, plus a learned residual to absorb deviations
    the analytical model cannot express.
    """

    def __init__(self, input_dim=13, width=256, num_blocks=5,
                 baseline_k=1e3, learn_k=True,
                 baseline_bounciness=0.5, learn_bounciness=True,
                 baseline_mass=1.0, learn_mass=True,
                 head_hidden=64):
        super().__init__()

        # --- backbone ---
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.LayerNorm(width),
            nn.GELU(),
        )

        backbone_layers = [nn.LayerNorm(width)]
        for _ in range(num_blocks, 1, -1):
            backbone_layers.append(ResBlock(width))
        self.backbone = nn.Sequential(*backbone_layers)

        self.head_trunk = nn.Sequential(
            nn.Linear(width, head_hidden),
            nn.LayerNorm(head_hidden),
            nn.GELU(),
        )
        self.head_out = nn.Linear(head_hidden, HEAD_OUT_DIM)

        # Physics parameters.
        self.k = nn.Parameter(
            torch.tensor(baseline_k, dtype=torch.float32),
            requires_grad=learn_k,
        )
        self.bounciness = nn.Parameter(
            torch.tensor(baseline_bounciness, dtype=torch.float32),
            requires_grad=learn_bounciness,
        )
        self.mass = nn.Parameter(
            torch.tensor(baseline_mass, dtype=torch.float32),
            requires_grad=learn_mass,
        )

    def forward(self, x, velocity):
        h = self.input_proj(x)
        h = self.backbone(h)

        raw = self.head_out(self.head_trunk(h))
        collision_logit, depth_raw, force_residual, normal_raw, lever = raw.split(
            [1, 1, 3, 3, 3], dim=-1
        )

        k_pos    = F.softplus(self.k)     if self.k.requires_grad else self.k
        mass_pos = F.softplus(self.mass)  if self.mass.requires_grad else self.mass
        mass_pos = torch.clamp(mass_pos, min=1e-6)
        bounciness = torch.sigmoid(self.bounciness)

        c = 2.0 * torch.sqrt(k_pos * mass_pos) * bounciness

        depth = torch.clamp(depth_raw, max=0.0)
        contact_normal = F.normalize(normal_raw, dim=-1, eps=1e-8)

        F_spring = -k_pos * depth
        vel_normal = (velocity * contact_normal).sum(dim=-1, keepdim=True)
        F_damping = -c * vel_normal
        F_mag = F.relu(F_spring + F_damping)

        force_normal = F_mag * contact_normal
        force_vec    = force_normal + force_residual
        torque_vec   = torch.cross(lever, force_vec, dim=-1)

        return {
            "collision_logit": collision_logit,
            "force":  force_vec,
            "torque": torque_vec,
            "aux": {
                "contact_normal": contact_normal,
                "lever":          lever,
                "depth":          depth,
                "k":              k_pos.detach(),
                "c":              c.detach(),
                "mass":           mass_pos.detach(),
                "F_mag":          F_mag,
                "F_spring":       F_spring,
                "F_damping":      F_damping,
                "force_normal":   force_normal,
                "force_residual": force_residual,
            },
        }


# --------------------------------------------------------------------------
# World-space vertex transform
# --------------------------------------------------------------------------
LIN_VEL_SLICE = slice(0, 3)
ANG_VEL_SLICE = slice(3, 6)
ROLL_SC_SLICE  = slice(7, 9)
PITCH_SC_SLICE = slice(9, 11)
YAW_SC_SLICE   = slice(11, 13)


def _rotmat_from_sincos(features: torch.Tensor) -> torch.Tensor:
    """[B, F] feature batch -> [B, 3, 3] rotation matrix.

    Z-Y-X intrinsic: R = Rz(yaw) Ry(pitch) Rx(roll).
    """
    sr, cr = features[:, 7:8],  features[:, 8:9]
    sp, cp = features[:, 9:10], features[:, 10:11]
    sy, cy = features[:, 11:12], features[:, 12:13]

    row0 = torch.cat([cy * cp,             cy * sp * sr - sy * cr,  cy * sp * cr + sy * sr], dim=-1)
    row1 = torch.cat([sy * cp,             sy * sp * sr + cy * cr,  sy * sp * cr - cy * sr], dim=-1)
    row2 = torch.cat([-sp,                 cp * sr,                 cp * cr               ], dim=-1)
    return torch.stack([row0, row1, row2], dim=1)


class VertexMLPWrench(nn.Module):
    VERT_GEOM_DIM = 16

    def __init__(self, state_dim: int = 13,
                 nodes_per_graph: int = num_vertices,
                 width: int = 256, num_blocks: int = 5,
                 encoder_dim: int = 256):
        super().__init__()
        self.N = nodes_per_graph
        self.state_dim = state_dim
        self.encoder_dim = encoder_dim

        # Encoder over the flattened all-vertex feature vector. Takes the
        # [B, N*16] block down to a fixed-size embedding before the head.
        flat_vert_dim = self.N * self.VERT_GEOM_DIM
        self.vertex_encoder = nn.Sequential(
            nn.Linear(flat_vert_dim, encoder_dim),
            nn.LayerNorm(encoder_dim),
            nn.GELU(),
            ResBlock(encoder_dim),
        )

        self.head = WrenchPredictor(
            input_dim=state_dim + encoder_dim,
            width=width, num_blocks=num_blocks,
        )

    def forward(self, features, vertex_pos, body_position=None):
        B = features.size(0)
        N = vertex_pos.size(0)

        R = _rotmat_from_sincos(features)

        # All vertices, transformed into world frame.
        vp_local = vertex_pos.unsqueeze(0).expand(B, N, 3)         # [B, N, 3]
        vp_world = torch.einsum("bij,bnj->bni", R, vp_local)

        if body_position is not None:
            vp_world = vp_world + body_position.unsqueeze(1)

        v_lin = features[:, LIN_VEL_SLICE].unsqueeze(1).expand(B, N, 3)
        v_ang = features[:, ANG_VEL_SLICE].unsqueeze(1).expand(B, N, 3)
        vp_world, vp_local, v_lin, v_ang = select_vertices(vp_world, vp_local, v_lin, v_ang, 502)
        N = vp_world.size(1)
        world_z = vp_world[..., 2:3]                               # [B, N, 1]

        vertex_vel = v_lin + torch.cross(v_ang, vp_world, dim=-1)

        vert_feats = torch.cat([
            vp_local, vp_world, world_z, vertex_vel, v_lin, v_ang
        ], dim=-1)                                                 # [B, N, 16]

        # Flatten all vertices, then encode down to a fixed embedding.
        flat_verts = vert_feats.reshape(B, N * self.VERT_GEOM_DIM)  # [B, N*16]
        graph_embed = self.vertex_encoder(flat_verts)              # [B, encoder_dim]

        x = torch.cat([features, graph_embed], dim=-1)
        return self.head(x, velocity=features[:, LIN_VEL_SLICE])


def select_vertices(vertex_pos_world, vp_local, v_lin, v_ang, num_vertices):
    # vertex_pos_world: [B, N, 3] -> [B, num_vertices, 3]
    order = vertex_pos_world[..., 1].argsort(dim=-1)        # [B, N]
    order = order[..., :num_vertices]                        # [B, num_vertices]
    idx = order.unsqueeze(-1).expand(-1, -1, 3)              # [B, num_vertices, 3]
    return torch.gather(vertex_pos_world, 1, idx),torch.gather(vp_local, 1, idx), torch.gather(v_lin, 1, idx), torch.gather(v_ang, 1, idx)

# --- usage in the module ---

def make_fast_predictor(input_dim=13, width=256, num_blocks=5,
                        encoder_dim=256):
    model = VertexMLPWrench(
        state_dim=input_dim,
        nodes_per_graph=502,
        width=width, num_blocks=num_blocks,
        encoder_dim=encoder_dim,
    )
    return model

In [ ]:
model = make_fast_predictor().to(device)

Print the model

In [ ]:

print(f"Model: {model}")
print(f"Backbone: {model.head.backbone}")
print(f"Head trunk: {model.head.head_trunk}")
print(f"Head out: {model.head.head_out}")

Model: VertexMLPWrench(
  (vertex_encoder): Sequential(
    (0): Linear(in_features=8032, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): ResBlock(
      (act): ReLU(inplace=True)
      (block): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Linear(in_features=256, out_features=1024, bias=True)
        (2): ReLU(inplace=True)
        (3): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (4): Linear(in_features=1024, out_features=256, bias=True)
        (5): ReLU(inplace=True)
      )
    )
  )
  (head): WrenchPredictor(
    (input_proj): Sequential(
      (0): Linear(in_features=269, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
    )
    (backbone): Sequential(
      (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (1): ResBlock(
        (act)

Check the model

In [ ]:
cpu_model = make_fast_predictor()
features = torch.zeros([1, 13])
vp = torch.tensor(vertice_positions, dtype=torch.float32)
out = cpu_model(features, vertex_pos=vp)


In [ ]:
model.eval()
with torch.no_grad():
    # Single-sample sanity check: 1 sample of 13-D state.
    features = torch.zeros([1, 13], device=device)
    vp = torch.tensor(vertice_positions, dtype=torch.float32, device=device)
    out = model(features, vertex_pos=vp)
    print("force shape :", tuple(out["force"].shape))
    print("torque shape:", tuple(out["torque"].shape))


force shape : (1, 3)
torque shape: (1, 3)


### Benchmarking

Benchmark the evaluation speed of the model

In [ ]:
NUM_BENCHMARKS = 1000
import time
import torch._logging
torch._logging.set_logs(recompiles=True, graph_breaks=True)

features = torch.rand((1, 13), device=device)
vp = torch.tensor(vertice_positions, dtype=torch.float32, device=device)

results = []
benchmark_model = make_fast_predictor().to(device).eval()
benchmark_model = torch.compile(benchmark_model)
benchmark_model.eval()

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

    with torch.inference_mode():
        # warmup
        for _ in tqdm(range(1000), desc="Warmup"):
            benchmark_model(features, vertex_pos=vp)
        print("=====Warmup completed======")

        for _ in tqdm(range(NUM_BENCHMARKS), desc="Benchmark"):
            torch.cuda.synchronize()
            start = time.perf_counter()
            benchmark_model(features, vertex_pos=vp)
            torch.cuda.synchronize()
            end = time.perf_counter()
            results.append((end - start) * 1000)

    torch.backends.cudnn.benchmark = False
    per_pred_ms = sum(results) / NUM_BENCHMARKS
    print(f"Per prediction: {per_pred_ms:.2f} ms")


Warmup:   0%|          | 0/1000 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:322: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
Warmup: 100%|██████████| 1000/1000 [00:14<00:00, 67.60it/s]


=====Warmup completed======


Benchmark: 100%|██████████| 1000/1000 [00:01<00:00, 894.35it/s]

Per prediction: 1.10 ms


In [ ]:
if torch.cuda.is_available():
    print(f"\n=== Benchmark Results ({device}) ===")
    print(f"Max: {max(results)} ms")
    print(f"Min: {min(results)} ms")
    print(f"Avg: {sum(results) / len(results)} ms")
    print(f"Median: {sorted(results)[len(results) // 2]} ms")
    print("Results:")
    print(results)



=== Benchmark Results (cuda) ===
Max: 1.9166390002283151 ms
Min: 1.0281079994456377 ms
Avg: 1.0959556640063965 ms
Median: 1.0560000000623404 ms
Results:
[1.9166390002283151, 1.0916860001088935, 1.067127000169421, 1.0596519996397546, 1.0632729999997537, 1.0505229993214016, 1.053187999787042, 1.0584550000203308, 1.0497639996174257, 1.0501890001251013, 1.0493929994481732, 1.0623620000842493, 1.0478889998921659, 1.0643470004652045, 1.0520599998926627, 1.0538320002524415, 1.0516359998291591, 1.0586300004433724, 1.0568059997240198, 1.066215999344422, 1.0548330001256545, 1.0541909996391041, 1.0574220004855306, 1.0497509993001586, 1.0464650003996212, 1.051062000442471, 1.066885000000184, 1.0553480005910387, 1.0626319999573752, 1.051438000104099, 1.0487159997865092, 1.0496670001884922, 1.0520569994696416, 1.061566000316816, 1.0624710002957727, 1.0546960002102423, 1.0551360001045396, 1.0477979994902853, 1.0560290002104011, 1.048629999786499, 1.0779470003399183, 1.0494380003365222, 1.05634500050

### CUDA Graph Benchmark

Captures the forward pass as a CUDA graph so the ~40 per-kernel dispatches are replaced by a single `cuGraphLaunch` on every call.

In [ ]:
if torch.cuda.is_available():
    # Fixed batch size for the captured graph.
    B = 1

    cuda_graph_model = make_fast_predictor().to(device).eval()

    # Static tensors — these addresses are baked into the captured graph.
    # Shapes must exactly match what the model is replayed with later.
    static_features   = torch.zeros(B, 13, device=device)
    static_vertex_pos = torch.tensor(vertice_positions, dtype=torch.float32,
                                     device=device)

    # --- Warmup before capture — must run on a side stream ---
    warmup_stream = torch.cuda.Stream()
    warmup_stream.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(warmup_stream):
        for _ in tqdm(range(1000), desc="CUDA graph warmup"):
            with torch.inference_mode():
                _ = cuda_graph_model(static_features, vertex_pos=static_vertex_pos)
    torch.cuda.current_stream().wait_stream(warmup_stream)
    torch.cuda.synchronize()

    # --- Capture ---
    cuda_graph = torch.cuda.CUDAGraph()
    with torch.inference_mode(), torch.cuda.graph(cuda_graph):
        static_output = cuda_graph_model(static_features, vertex_pos=static_vertex_pos)
    print("CUDA graph captured.")

    # --- Benchmark ---
    # Pre-generate inputs OUTSIDE the timing loop.
    inputs = [torch.rand(B, 13, device=device) for _ in range(NUM_BENCHMARKS)]
    torch.cuda.synchronize()

    cuda_graph_results = []
    with torch.inference_mode():
        for x in tqdm(inputs, desc="CUDA graph benchmark"):
            torch.cuda.synchronize()
            start = time.perf_counter()
            static_features.copy_(x)
            cuda_graph.replay()
            torch.cuda.synchronize()
            end = time.perf_counter()
            cuda_graph_results.append((end - start) * 1000)

    print(f"\n=== CUDA Graph Benchmark Results ({device}) ===")
    print(f"Max:    {max(cuda_graph_results):.4f} ms")
    print(f"Min:    {min(cuda_graph_results):.4f} ms")
    print(f"Avg:    {sum(cuda_graph_results) / len(cuda_graph_results):.4f} ms")
    print(f"Median: {sorted(cuda_graph_results)[len(cuda_graph_results) // 2]:.4f} ms")
    print(f"\nSpeedup over baseline: "
          f"{(sum(results) / len(results)) / (sum(cuda_graph_results) / len(cuda_graph_results)):.1f}x")


CUDA graph warmup: 100%|██████████| 1000/1000 [00:02<00:00, 449.76it/s]


CUDA graph captured.


CUDA graph benchmark: 100%|██████████| 1000/1000 [00:00<00:00, 2811.48it/s]


=== CUDA Graph Benchmark Results (cuda) ===
Max:    0.9449 ms
Min:    0.3119 ms
Avg:    0.3409 ms
Median: 0.3160 ms

Speedup over baseline: 3.2x


## 4. Loss

Huber (smooth-L1) loss replaces L1.  Near zero it's quadratic (smooth gradients,
won't over-punish tiny residuals); far from zero it's linear (robust to the
occasional outlier).  `delta=1.0` is in *normalized* target units so it's
roughly one standard deviation of the target.

**Energy-conservation penalty.**  For each collision sample we integrate one
timestep forward using the *predicted* wrench and check whether the body's
kinetic energy would grow beyond `e^2 * KE_before` (where `e` is the
coefficient of restitution).  Any excess is squared and added to the loss,
so the term is zero for physically admissible predictions and grows smoothly
when the model would inject energy — the exact failure mode you were seeing
at low collision speeds.  You control it with `w_energy`, `dt`, `mass`,
`inertia_diag`, and `restitution` when constructing `WrenchLoss`.


In [ ]:
class WrenchLoss(nn.Module):
    """Physics-structured loss for a Hooke (linear elastic) contact model.

    Components:
        - BCE on collision_logit (with pos_weight for class imbalance).
        - Masked Huber on force and torque, operating in PHYSICAL units.
        - Soft penalty on f_n < 0 (Signorini violation).
        - Soft penalty on energy gain during collision (restitution-aware),
          using a BOUNDED log-based term so large violations at init don't
          produce enormous gradients that kill regression learning.

    Energy-conservation term
    ------------------------
    Given a predicted force F and torque tau applied over one timestep dt to a
    body with mass m and (diagonal) inertia I, the post-step velocities are
        v'   = v   + (F   / m) * dt
        w'   = w   + (I^-1 tau) * dt
    and the kinetic energy is  KE = 0.5 m |v|^2 + 0.5 w^T I w.
    For a real collision with coefficient of restitution e in [0, 1] we expect
        KE'  <=  e^2 * KE_before.
    We define the energy ratio
        r = KE' / (e^2 * KE_before)
    and penalise log(max(r, 1))^2. This is zero when r <= 1 (admissible),
    grows like (log r)^2 when r > 1, and its gradient in r is bounded — so a
    random-init network that predicts wildly wrong forces at epoch 0 won't
    produce an exploding energy gradient that pushes the model into the
    degenerate F≈0 basin.

    Warmup: w_energy typically starts at 0 and is ramped up over several
    epochs by the training loop via `set_energy_weight`, so the regression
    heads (force, torque) get to learn first before conservation pressure
    kicks in.

    Because targets are NOT pre-normalised, you may need to set w_force and
    w_torque to bring the two regression terms to comparable magnitude. A good
    heuristic is to set:
        w_force  ~ 1 / (force_rms_in_physical_units)
        w_torque ~ 1 / (torque_rms_in_physical_units)
    so both contribute roughly equally early in training.
    """
    def __init__(self,
                 w_force=1.0, w_torque=1.0, w_collision=1.0,
                 w_energy=0.0,
                 huber_delta=1.0, pos_weight=None,
                 dt=0.24, mass=1.0, inertia_diag=(1.0/6.0, 1.0/6.0, 1.0/6.0),
                 restitution=0.5):
        super().__init__()
        self.w_force     = w_force
        self.w_torque    = w_torque
        self.w_collision = w_collision
        self.w_energy    = w_energy
        self.huber_delta = huber_delta
        self.dt          = float(dt)
        self.mass        = float(mass)
        self.restitution = float(restitution)
        # Inertia tensor (diagonal) for a unit cube by default: I = (1/6) m a^2
        # with m=1, a=1. Override via the constructor to match your simulated body.
        self.register_buffer(
            "inertia_diag",
            torch.tensor(inertia_diag, dtype=torch.float32),
        )
        # pos_weight is a tensor; register as buffer so .to(device) moves it.
        if pos_weight is not None and not torch.is_tensor(pos_weight):
            pos_weight = torch.tensor(float(pos_weight))
        self.register_buffer(
            "pos_weight",
            pos_weight if pos_weight is not None else torch.tensor(1.0),
        )
        self._has_pos_weight = pos_weight is not None

    def set_energy_weight(self, w):
        """Runtime hook for the training loop's warmup schedule."""
        self.w_energy = float(w)

    def _kinetic_energy(self, v, w):
        """KE = 0.5 m |v|^2 + 0.5 w^T I w for diagonal I. Shapes: (B,3)."""
        ke_lin = 0.5 * self.mass * (v * v).sum(dim=-1, keepdim=True)
        ke_rot = 0.5 * (self.inertia_diag * w * w).sum(dim=-1, keepdim=True)
        return ke_lin + ke_rot

    def forward(self, preds, targets):
        mask   = targets["is_collision"]                   # (B, 1)
        n_coll = mask.sum().clamp_min(1.0)

        # --- collision BCE ---
        loss_collision = F.binary_cross_entropy_with_logits(
            preds["collision_logit"], mask.float(),
            pos_weight=self.pos_weight if self._has_pos_weight else None,
            reduction="mean",
        )
        # --- masked Huber on force and torque (physical units) ---
        raw_f = F.huber_loss(preds["force"],  targets["force"],
                             reduction="none", delta=self.huber_delta)  # (B, 3)
        raw_t = F.huber_loss(preds["torque"], targets["torque"],
                             reduction="none", delta=self.huber_delta)  # (B, 3)
        loss_force  = (raw_f.sum(dim=-1, keepdim=True) * mask).sum() / n_coll
        loss_torque = (raw_t.sum(dim=-1, keepdim=True) * mask).sum() / n_coll

        # --- energy-conservation penalty (bounded, log-based) ---
        # Integrate one step using the predicted wrench and compare KE before vs after.
        # Only collision samples contribute (non-contact steps have F=tau=0 anyway).
        v = targets["lin_vel"]                              # (B, 3)
        w = targets["ang_vel"]                              # (B, 3)
        f_pred = preds["force"]                             # (B, 3)
        t_pred = preds["torque"]                            # (B, 3)

        v_next = v + (f_pred / self.mass) * self.dt
        # Diagonal inertia -> element-wise divide
        w_next = w + (t_pred / self.inertia_diag) * self.dt

        ke_before = self._kinetic_energy(v, w)              # (B, 1)
        ke_after  = self._kinetic_energy(v_next, w_next)    # (B, 1)

        # Log-ratio penalty:
        #   r = ke_after / (e^2 * ke_before + eps),  penalty = max(log r, 0)^2
        # Bounded gradient in F: d/dF log(ke_after) scales as 1/ke_after, so at
        # init where ke_after is huge the gradient is SMALL — the opposite of
        # (ke_after - budget)^2, which has gradient proportional to ke_after.
        eps       = 1e-6
        ke_budget = (self.restitution ** 2) * ke_before
        log_ratio = torch.log(ke_after + eps) - torch.log(ke_budget + eps)
        excess_log  = F.relu(log_ratio)                     # zero when admissible
        loss_energy = ((excess_log ** 2) * mask).sum() / n_coll

        total = (self.w_force     * loss_force
               + self.w_torque    * loss_torque
               + self.w_collision * loss_collision
               + 0   * loss_energy)

        # Diagnostic: fraction of collision samples that currently violate conservation,
        # plus the geometric-mean ratio so you can see HOW MUCH they violate by.
        with torch.no_grad():
            violating = ((excess_log > 0).float() * mask).sum() / n_coll
            # Mean log-ratio over collision samples (in log space so it's well-behaved)
            mean_log_ratio = (log_ratio * mask).sum() / n_coll

        return total, {
            "force":             loss_force.item(),
            "torque":            loss_torque.item(),
            "collision":         loss_collision.item(),
            "energy":            loss_energy.item(),
            "energy_violating":  violating.item(),
            "energy_mean_logr":  mean_log_ratio.item(),
            "w_energy":          self.w_energy,
            "total":             total.item(),
            "active_collisions": n_coll.item(),
            "k":                 preds["aux"]["k"].item(),
        }

## Training

In [ ]:
def train_model(model, train_loader, val_loader, train_dataset,
                epochs=200, lr=1e-4, weight_decay=1e-4,
                w_force=1.0, w_torque=1.0, w_collision=0.5,
                # Energy-conservation warmup schedule.
                # Rationale: starting with w_energy>0 produces enormous gradients
                # at init (ke_after is huge for a random-init network) and pushes
                # the model into the degenerate F≈0 basin, where regression loss
                # plateaus at force_rms. Warming up from 0 lets the force/torque
                # heads learn a reasonable solution first; only then do we
                # gently tighten conservation.
                w_energy_max=0.1, energy_warmup_start=20, energy_warmup_epochs=30,
                dt=0.24, mass=1.0, inertia_diag=(1.0/6.0, 1.0/6.0, 1.0/6.0),
                restitution=0.5):
    model.to(device)

    # Vertex positions are constant across all samples — build once.
    vp = torch.tensor(vertice_positions, dtype=torch.float32, device=device)

    # Class-imbalance weight for BCE: #no-contact / #contact on train set.
    collisions = train_dataset.collisions.squeeze(-1).bool()
    n_pos = int(collisions.sum().item())
    n_neg = int((~collisions).sum().item())
    pos_weight = (n_neg / max(n_pos, 1)) if n_pos > 0 else 1.0
    print(f"BCE pos_weight = {pos_weight:.3f}  ({n_pos} contacts / {n_neg} non-contacts)")

    criterion = WrenchLoss(
        w_force=w_force, w_torque=w_torque, w_collision=w_collision,
        w_energy=0.0,  # ramped up by the warmup schedule below
        dt=dt, mass=mass, inertia_diag=inertia_diag, restitution=restitution,
        huber_delta=1.0, pos_weight=pos_weight,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=20
    )

    def energy_weight_at(epoch):
        """Linear ramp from 0 to w_energy_max over [start, start+epochs)."""
        if epoch < energy_warmup_start:
            return 0.0
        if energy_warmup_epochs <= 0:
            return float(w_energy_max)
        frac = (epoch - energy_warmup_start) / float(energy_warmup_epochs)
        return float(w_energy_max) * min(max(frac, 0.0), 1.0)

    best_val_loss = float('inf')
    loss_keys = ['total', 'force', 'torque', 'collision',
                 'energy', 'energy_violating', 'energy_mean_logr']

    for epoch in tqdm(range(epochs)):
        criterion.set_energy_weight(energy_weight_at(epoch))

        # --- Train ---
        model.train()
        train_losses = {k: 0.0 for k in loss_keys}
        for features, targets in train_loader:
            features = features.to(device)                       # [B, 13]
            targets  = {k: v.to(device) for k, v in targets.items()}
            out = model(
                features,
                vertex_pos=vp,
                body_position=targets["self_position"],
            )
            optimizer.zero_grad()

            loss, components = criterion(out, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            for k in loss_keys:
                train_losses[k] += components[k]
        for k in loss_keys:
            train_losses[k] /= len(train_loader)

        # --- Validate ---
        model.eval()
        val_losses = {k: 0.0 for k in loss_keys}
        with torch.no_grad():
            for features, targets in val_loader:
                features = features.to(device)
                targets  = {k: v.to(device) for k, v in targets.items()}
                _, components = criterion(
                    model(
                        features,
                        vertex_pos=vp,
                        body_position=targets["self_position"],
                    ),
                    targets,
                )
                for k in loss_keys:
                    val_losses[k] += components[k]
        for k in loss_keys:
            val_losses[k] /= len(val_loader)

        scheduler.step(val_losses['total'])

        if val_losses['total'] < best_val_loss:
            best_val_loss = val_losses['total']
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss':        best_val_loss,
            }, 'wrench_model_best.pth')

        if (epoch + 1) % 10 == 0:
            lr_now = optimizer.param_groups[0]['lr']
            k_now  = components.get('k', float('nan'))
            we_now = criterion.w_energy
            print(f"Epoch {epoch+1}/{epochs} | LR: {lr_now:.2e} | k={k_now:.1f} | w_energy={we_now:.4f}")
            print(f"  Train: total={train_losses['total']:.4f} "
                  f"f={train_losses['force']:.4f} t={train_losses['torque']:.4f} "
                  f"c={train_losses['collision']:.4f} "
                  f"e={train_losses['energy']:.4f} vio={train_losses['energy_violating']:.2f} "
                  f"logr={train_losses['energy_mean_logr']:+.3f}")
            print(f"  Val:   total={val_losses['total']:.4f} "
                  f"f={val_losses['force']:.4f} t={val_losses['torque']:.4f} "
                  f"c={val_losses['collision']:.4f} "
                  f"e={val_losses['energy']:.4f} vio={val_losses['energy_violating']:.2f} "
                  f"logr={val_losses['energy_mean_logr']:+.3f}")

    return model


### Execute training

In [ ]:
model = train_model(model, train_loader, val_loader, full_dataset,
                    epochs=200, lr=1e-3,
                    w_force=1.0, w_torque=1.0, w_collision=0.5)


BCE pos_weight = 0.920  (1118967 contacts / 1029339 non-contacts)


  5%|▌         | 10/200 [05:49<2:03:08, 38.89s/it]

Epoch 10/200 | LR: 1.00e-03 | k=999.5 | w_energy=0.0000
  Train: total=95.2124 f=75.6736 t=19.5063 c=0.0650 e=31.4635 vio=1.00 logr=+5.022
  Val:   total=94.9814 f=75.6415 t=19.3099 c=0.0600 e=31.1901 vio=1.00 logr=+5.024


 10%|█         | 20/200 [14:12<2:24:58, 48.32s/it]

Epoch 20/200 | LR: 1.00e-03 | k=997.8 | w_energy=0.0000
  Train: total=63.9542 f=48.0979 t=15.8385 c=0.0355 e=31.7028 vio=1.00 logr=+5.031
  Val:   total=65.7086 f=49.2510 t=16.4404 c=0.0345 e=31.4822 vio=1.00 logr=+4.995


...

Epoch 180/200 | LR: 2.50e-04 | k=971.0 | w_energy=0.1000
  Train: total=11.4888 f=7.2474 t=4.2369 c=0.0091 e=31.7328 vio=1.00 logr=+5.024
  Val:   total=23.5529 f=16.5950 t=6.9536 c=0.0085 e=31.8553 vio=1.00 logr=+5.032


 95%|█████████▌| 190/200 [2:05:51<08:24, 50.47s/it]

Epoch 190/200 | LR: 2.50e-04 | k=971.0 | w_energy=0.1000
  Train: total=11.0287 f=6.8966 t=4.1277 c=0.0088 e=31.7409 vio=1.00 logr=+5.024
  Val:   total=23.2110 f=16.1844 t=7.0220 c=0.0092 e=32.0396 vio=1.00 logr=+5.050


100%|██████████| 200/200 [2:14:22<00:00, 40.31s/it]

Epoch 200/200 | LR: 2.50e-04 | k=971.0 | w_energy=0.1000
  Train: total=10.7098 f=6.6739 t=4.0315 c=0.0087 e=31.7434 vio=1.00 logr=+5.024
  Val:   total=22.9804 f=16.1941 t=6.7821 c=0.0085 e=31.7411 vio=1.00 logr=+5.024


## Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, loader, dataset, device="cuda",
             rel_floor_force=0.05, rel_floor_torque=0.05):
    """Evaluate in physical units (targets were not normalised).

    Args:
        dataset: the *underlying* ContactDataset (not a Subset). Kept for API
                 symmetry; no target stats are needed since targets are physical.
        rel_floor_{force,torque}: targets with magnitude below this (in
                 physical units) are excluded from the relative-error stats
                 to avoid division-by-near-zero blow-up.
    """
    model.eval()
    all_abs_f, all_abs_t = [], []
    all_rel_f, all_rel_t = [], []
    all_coll_correct     = []

    # Physics-diagnostic accumulators (frictionless model: only f_n sign check)
    n_fn_neg = 0
    n_total  = 0

    vp = torch.tensor(vertice_positions, dtype=torch.float32, device=device)

    for features, targets in loader:
        features = features.to(device)
        f_tgt = targets["force"].to(device)
        t_tgt = targets["torque"].to(device)
        c_tgt = targets["is_collision"].to(device).squeeze(-1).bool()
        body_position = targets["self_position"].to(device)

        preds = model(features, vertex_pos=vp, body_position=body_position)

        f_pred = preds["force"]
        t_pred = preds["torque"]
        c_pred = (torch.sigmoid(preds["collision_logit"]).squeeze(-1) > 0.5)

        all_coll_correct.append((c_pred == c_tgt).float().cpu())

        if c_tgt.any():
            f_pred_c = f_pred[c_tgt]
            f_tgt_c  = f_tgt[c_tgt]
            t_pred_c = t_pred[c_tgt]
            t_tgt_c  = t_tgt[c_tgt]

            abs_f = (f_pred_c - f_tgt_c).norm(dim=-1)
            abs_t = (t_pred_c - t_tgt_c).norm(dim=-1)
            all_abs_f.append(abs_f.cpu())
            all_abs_t.append(abs_t.cpu())

            f_norm = f_tgt_c.norm(dim=-1)
            t_norm = t_tgt_c.norm(dim=-1)
            mask_f = f_norm > rel_floor_force
            mask_t = t_norm > rel_floor_torque
            if mask_f.any():
                rel_f = (f_pred_c[mask_f] - f_tgt_c[mask_f]).norm(dim=-1) / f_norm[mask_f]
                all_rel_f.append(rel_f.cpu())
            if mask_t.any():
                rel_t = (t_pred_c[mask_t] - t_tgt_c[mask_t]).norm(dim=-1) / t_norm[mask_t]
                all_rel_t.append(rel_t.cpu())

            # Physics diagnostic: how often the Hooke-plus-residual allows f_n < 0.
            f_n_c = preds["aux"]["force_residual"][c_tgt]
            n_fn_neg += int((f_n_c < 0).sum().item())
            n_total  += int(c_tgt.sum().item())

    abs_f = torch.cat(all_abs_f) if all_abs_f else torch.empty(0)
    abs_t = torch.cat(all_abs_t) if all_abs_t else torch.empty(0)
    rel_f = torch.cat(all_rel_f) if all_rel_f else torch.empty(0)
    rel_t = torch.cat(all_rel_t) if all_rel_t else torch.empty(0)
    coll_acc = torch.cat(all_coll_correct).mean().item()

    def fmt_pct(e):
        if e.numel() == 0:
            return "n/a"
        return (f"p50={e.median():.1%}  p90={e.quantile(0.9):.1%}  "
                f"p99={e.quantile(0.99):.1%}")
    def fmt_abs(e):
        if e.numel() == 0:
            return "n/a"
        return (f"p50={e.median():.4f}  p90={e.quantile(0.9):.4f}  "
                f"p99={e.quantile(0.99):.4f}")

    print("─" * 60)
    print(f"Collision accuracy : {coll_acc:.3%}")
    print(f"Force  abs err     : {fmt_abs(abs_f)}")
    print(f"Torque abs err     : {fmt_abs(abs_t)}")
    print(f"Force  rel err     : {fmt_pct(rel_f)}  "
          f"(on {rel_f.numel()} / {abs_f.numel()} samples above floor)")
    print(f"Torque rel err     : {fmt_pct(rel_t)}  "
          f"(on {rel_t.numel()} / {abs_t.numel()} samples above floor)")
    if n_total > 0:
        print(f"Physics violations : f_n<0 in {n_fn_neg}/{n_total} "
              f"({100*n_fn_neg/n_total:.2f}%)")
    print("─" * 60)

    return {
        "collision_acc":  coll_acc,
        "force_abs_p50":  abs_f.median().item() if abs_f.numel() else float("nan"),
        "force_abs_p99":  abs_f.quantile(0.99).item() if abs_f.numel() else float("nan"),
        "torque_abs_p50": abs_t.median().item() if abs_t.numel() else float("nan"),
        "torque_abs_p99": abs_t.quantile(0.99).item() if abs_t.numel() else float("nan"),
        "fn_neg_rate":    (n_fn_neg / n_total) if n_total > 0 else float("nan"),
    }


## Print 100 data points

In [ ]:
@torch.no_grad()
def print_predictions(model, loader, n=100):
    """Print n predictions vs ground truth from the loader."""
    model.eval()
    all_f_pred, all_f_tgt = [], []
    all_t_pred, all_t_tgt = [], []
    all_coll_pred, all_coll_tgt = [], []

    vp = torch.tensor(vertice_positions, dtype=torch.float32, device=device)

    for features, targets in loader:
        features = features.to(device)
        body_position = targets["self_position"].to(device)
        preds = model(features, vertex_pos=vp, body_position=body_position)

        all_f_pred.append(preds["force"].cpu())
        all_t_pred.append(preds["torque"].cpu())
        all_f_tgt.append(targets["force"])
        all_t_tgt.append(targets["torque"])
        all_coll_pred.append(torch.sigmoid(preds["collision_logit"]).cpu())
        all_coll_tgt.append(targets["is_collision"])
        collected = sum(x.shape[0] for x in all_f_pred)
        if collected >= n:
            break

    f_pred = torch.cat(all_f_pred)[:n]
    f_tgt  = torch.cat(all_f_tgt)[:n]
    t_pred = torch.cat(all_t_pred)[:n]
    t_tgt  = torch.cat(all_t_tgt)[:n]
    c_pred = torch.cat(all_coll_pred)[:n].squeeze(-1)
    c_tgt  = torch.cat(all_coll_tgt)[:n].squeeze(-1)

    header = (f"{'#':>4s}  {'coll':>5s} {'pred':>5s}  "
              f"{'force_pred':>30s}  {'force_true':>30s}  "
              f"{'torque_pred':>30s}  {'torque_true':>30s}  ")
    print(header)
    print("─" * len(header))
    for i in range(n):
        cp = f"{c_pred[i]:.2f}"
        ct = f"{int(c_tgt[i].item())}"
        fp = f"[{f_pred[i,0]:8.3f}, {f_pred[i,1]:8.3f}, {f_pred[i,2]:8.3f}]"
        ft = f"[{f_tgt[i,0]:8.3f}, {f_tgt[i,1]:8.3f}, {f_tgt[i,2]:8.3f}]"
        tp = f"[{t_pred[i,0]:8.4f}, {t_pred[i,1]:8.4f}, {t_pred[i,2]:8.4f}]"
        tt = f"[{t_tgt[i,0]:8.4f}, {t_tgt[i,1]:8.4f}, {t_tgt[i,2]:8.4f}]"
        print(f"{i:4d}  {ct:>5s} {cp:>5s}  {fp:>30s}  {ft:>30s}  {tp:>30s}  {tt:>30s}")


print_predictions(model, val_loader, n=100)


   #   coll  pred                      force_pred                      force_true                     torque_pred                     torque_true  
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   0      0  0.00  [   0.003,    0.184,  -54.616]  [   0.000,    0.000,    0.000]  [  1.6209,   4.3160,   0.0147]  [  0.0000,   0.0000,   0.0000]
   1      1  0.99  [  -0.005,   -0.032,  120.788]  [  -0.000,    0.000,  120.035]  [-57.8874,  31.6484,   0.0059]  [-57.6294,  32.2025,  -0.0000]
   2      0  0.00  [   0.465,    0.029,  169.704]  [   0.000,    0.000,    0.000]  [  1.9622,  -4.2632,  -0.0046]  [  0.0000,   0.0000,   0.0000]
   3      0  0.00  [   0.425,    0.207,   43.208]  [   0.000,    0.000,    0.000]  [  1.2809,   1.0049,  -0.0174]  [  0.0000,   0.0000,   0.0000]
   4      0  0.00  [  -0.247,    0.510,  -13.842]  [   0.000,    0.000,    0.000]  [  0.1467,   0.6694,   0.0221]  [  0.

Download checkpoint (Colab)

In [ ]:
if is_colab():
    from google.colab import files
    files.download("wrench_model_best.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>